In [ ]:
import json

INPUT = "processed_document.jsonl"

documents = []

with open(INPUT, "r", encoding="utf-8") as f:
    for line in f:
        documents.append(json.loads(line))

print(len(documents))
print(documents[0].keys())

85
dict_keys(['text', 'metadata'])


In [ ]:
import re

def normalize_whitespace(text):

    text = text.replace("\r", "\n")

    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


for doc in documents:
    doc["text"] = normalize_whitespace(doc["text"])

In [ ]:
def remove_layout_tags(text):

    text = re.sub(
        r"<\|det\|>.*?<\|/det\|>",
        "",
        text,
        flags=re.DOTALL
    )

    return text


for doc in documents:
    doc["text"] = remove_layout_tags(doc["text"])

In [ ]:
def remove_html(text):

    text = re.sub(r"<[^>]+>", "", text)

    return text


for doc in documents:
    doc["text"] = remove_html(doc["text"])

In [ ]:
def fix_hyphenation(text):

    text = re.sub(
        r"(\w)-\n(\w)",
        r"\1-\2",
        text
    )

    return text


for doc in documents:
    doc["text"] = fix_hyphenation(doc["text"])

In [ ]:
def clean_spacing(text):

    text = re.sub(r"[ ]{2,}", " ", text)

    text = re.sub(r"\n +", "\n", text)

    text = re.sub(r" +\n", "\n", text)

    return text.strip()


for doc in documents:
    doc["text"] = clean_spacing(doc["text"])

In [ ]:
HEADER_PATTERNS = [
    r"World Health Organization",
    r"WHO recommendations for care of the preterm or low-birth-weight infant",
]

def remove_headers(text):

    for p in HEADER_PATTERNS:

        text = re.sub(
            p,
            "",
            text,
            flags=re.IGNORECASE
        )

    return text


for doc in documents:
    doc["text"] = remove_headers(doc["text"])

In [ ]:
for doc in documents:

    doc["text"] = normalize_whitespace(doc["text"])

In [ ]:
for i in range(3):

    print("="*80)

    print(documents[i]["text"][:1000])

WHO recommendations for care of the preterm or low birth weight infant ISBN 978-92-4-005826-2 (electronic version) ISBN 978-92-4-005827-9 (print version) © 2022 Some rights reserved. This work is available under the Creative Commons Attribution-NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/licenses/by-nc-sa/3.0/igo). Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial purposes, provided the work is appropriately cited, as indicated below. In any use of this work, there should be no suggestion that WHO endorses any specific organization, products or services. The use of the WHO logo is not permitted. If you adapt the work, then you must license your work under the same or equivalent Creative Commons licence. If you create a translation of this work, you should add the following disclaimer along with the suggested citation: "This translation was not created by the (WHO). WHO is not responsible for 

In [ ]:
OUTPUT = "processed_document_clean.jsonl"

with open(OUTPUT, "w", encoding="utf-8") as f:

    for doc in documents:

        f.write(
            json.dumps(
                doc,
                ensure_ascii=False
            )
            + "\n"
        )

print("Saved:", OUTPUT)

Saved: processed_document_clean.jsonl


In [ ]:
from pprint import pprint

for i, doc in enumerate(documents[:5]):

    print("=" * 100)
    print(f"Chunk {i}\n")

    pprint(doc["metadata"])

    print("\nTEXT:\n")
    print(doc["text"][:1500])

Chunk 0

{'chapter': None, 'section': None, 'subsection': None}

TEXT:

WHO recommendations for care of the preterm or low birth weight infant ISBN 978-92-4-005826-2 (electronic version) ISBN 978-92-4-005827-9 (print version) © 2022 Some rights reserved. This work is available under the Creative Commons Attribution-NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/licenses/by-nc-sa/3.0/igo). Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial purposes, provided the work is appropriately cited, as indicated below. In any use of this work, there should be no suggestion that WHO endorses any specific organization, products or services. The use of the WHO logo is not permitted. If you adapt the work, then you must license your work under the same or equivalent Creative Commons licence. If you create a translation of this work, you should add the following disclaimer along with the suggested citation: "Th

In [ ]:
import re

patterns = {
    "OCR tag": r"<\|det\|>",
    "HTML tag": r"<table|<tr|<td|</table>|</tr>|</td>",
    "Multiple spaces": r" {2,}",
    "Many newlines": r"\n{3,}",
}

for name, pattern in patterns.items():

    count = sum(
        bool(re.search(pattern, doc["text"], re.IGNORECASE))
        for doc in documents
    )

    print(f"{name:20}: {count}")

OCR tag             : 0
HTML tag            : 0
Multiple spaces     : 0
Many newlines       : 0


In [ ]:
import json

INPUT = "processed_document_clean.jsonl"

documents = []

with open(INPUT, "r", encoding="utf-8") as f:
    for line in f:
        documents.append(json.loads(line))

print(len(documents))

85


In [ ]:
!pip install pysbd -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 4.2 MB/s eta 0:00:00


In [ ]:
import pysbd

segmenter = pysbd.Segmenter(
    language="en",
    clean=False
)

for doc in documents:

    doc["sentences"] = segmenter.segment(
        doc["text"]
    )

In [ ]:
from pprint import pprint

sample = documents[0]

print(sample["metadata"])

print()

for i, s in enumerate(sample["sentences"]):

    print(f"{i+1}. {s}")

{'chapter': None, 'section': None, 'subsection': None}

1. WHO recommendations for care of the preterm or low birth weight infant ISBN 978-92-4-005826-2 (electronic version) ISBN 978-92-4-005827-9 (print version) © 2022 Some rights reserved. 
2. This work is available under the Creative Commons Attribution-NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/licenses/by-nc-sa/3.0/igo). 
3. Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial purposes, provided the work is appropriately cited, as indicated below. 
4. In any use of this work, there should be no suggestion that WHO endorses any specific organization, products or services. 
5. The use of the WHO logo is not permitted. 
6. If you adapt the work, then you must license your work under the same or equivalent Creative Commons licence. 
7. If you create a translation of this work, you should add the following disclaimer along with the suggested ci

In [ ]:
num_sentences = [
    len(doc["sentences"])
    for doc in documents
]

print("Average :", sum(num_sentences)/len(num_sentences))
print("Max :", max(num_sentences))
print("Min :", min(num_sentences))

Average : 19.32941176470588
Max : 255
Min : 1


In [ ]:
OUTPUT = "sentence_split.jsonl"

with open(OUTPUT, "w", encoding="utf-8") as f:

    for doc in documents:

        f.write(
            json.dumps(
                doc,
                ensure_ascii=False
            ) + "\n"
        )

print("Saved:", OUTPUT)

Saved: sentence_split.jsonl


In [ ]:
import json

INPUT = "sentence_split.jsonl"

documents = []

with open(INPUT, "r", encoding="utf-8") as f:
    for line in f:
        documents.append(json.loads(line))

print(len(documents))

85


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "BAAI/bge-m3"
)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [ ]:
def token_length(text):

    return len(
        tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )

In [ ]:
MAX_TOKEN = 350

chunks = []

for doc in documents:

    current = []

    current_len = 0

    for sentence in doc["sentences"]:

        sent_len = token_length(sentence)

        if current_len + sent_len > MAX_TOKEN:

            chunks.append({

                "text": " ".join(current),

                "metadata": doc["metadata"]

            })

            current = []

            current_len = 0

        current.append(sentence)

        current_len += sent_len

    if current:

        chunks.append({

            "text": " ".join(current),

            "metadata": doc["metadata"]

        })

In [ ]:
print("Total chunks:", len(chunks))

Total chunks: 259


In [ ]:
lengths = [
    token_length(chunk["text"])
    for chunk in chunks
]

print("Average:", sum(lengths)/len(lengths))
print("Min:", min(lengths))
print("Max:", max(lengths))

Average: 248.996138996139
Min: 0
Max: 653


In [ ]:
from pprint import pprint

for chunk in chunks[:3]:

    pprint(chunk["metadata"])

    print()

    print(chunk["text"])

    print("="*80)

{'chapter': None, 'section': None, 'subsection': None}

WHO recommendations for care of the preterm or low birth weight infant ISBN 978-92-4-005826-2 (electronic version) ISBN 978-92-4-005827-9 (print version) © 2022 Some rights reserved.  This work is available under the Creative Commons Attribution-NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/licenses/by-nc-sa/3.0/igo).  Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial purposes, provided the work is appropriately cited, as indicated below.  In any use of this work, there should be no suggestion that WHO endorses any specific organization, products or services.  The use of the WHO logo is not permitted.  If you adapt the work, then you must license your work under the same or equivalent Creative Commons licence.  If you create a translation of this work, you should add the following disclaimer along with the suggested citation: "This transla

In [ ]:
OUTPUT = "sentence_chunk.jsonl"

with open(OUTPUT, "w", encoding="utf-8") as f:

    for chunk in chunks:

        f.write(
            json.dumps(
                chunk,
                ensure_ascii=False
            )
            + "\n"
        )

print("Saved:", OUTPUT)

Saved: sentence_chunk.jsonl


In [ ]:
import json

INPUT = "sentence_split.jsonl"

documents = []

with open(INPUT, "r", encoding="utf-8") as f:
    for line in f:
        documents.append(json.loads(line))

print(len(documents))

85


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

In [ ]:
def token_length(text):
    return len(
        tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )

In [ ]:
MAX_TOKEN = 350
OVERLAP_SENTENCES = 2

chunks = []

for doc in documents:

    sentences = doc["sentences"]

    start = 0

    while start < len(sentences):

        current = []
        current_len = 0
        end = start

        while end < len(sentences):

            sent = sentences[end]
            sent_len = token_length(sent)

            if current_len + sent_len > MAX_TOKEN:
                break

            current.append(sent)
            current_len += sent_len
            end += 1

        chunks.append({
            "text": " ".join(current),
            "metadata": doc["metadata"]
        })

        if end == len(sentences):
            break

        start = max(end - OVERLAP_SENTENCES, start + 1)

In [ ]:
for i in range(3):

    print("="*80)
    print(f"Chunk {i}")
    print(chunks[i]["text"])

Chunk 0
WHO recommendations for care of the preterm or low birth weight infant ISBN 978-92-4-005826-2 (electronic version) ISBN 978-92-4-005827-9 (print version) © 2022 Some rights reserved.  This work is available under the Creative Commons Attribution-NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/licenses/by-nc-sa/3.0/igo).  Under the terms of this licence, you may copy, redistribute and adapt the work for non-commercial purposes, provided the work is appropriately cited, as indicated below.  In any use of this work, there should be no suggestion that WHO endorses any specific organization, products or services.  The use of the WHO logo is not permitted.  If you adapt the work, then you must license your work under the same or equivalent Creative Commons licence.  If you create a translation of this work, you should add the following disclaimer along with the suggested citation: "This translation was not created by the (WHO). WHO is not re

In [ ]:
lengths = [
    token_length(c["text"])
    for c in chunks
]

print("Chunks:", len(chunks))
print("Average:", sum(lengths)/len(lengths))
print("Max:", max(lengths))
print("Min:", min(lengths))

Chunks: 327
Average: 243.18960244648318
Max: 350
Min: 0


In [ ]:
OUTPUT = "sentence_overlap_chunks.jsonl"

with open(OUTPUT, "w", encoding="utf-8") as f:

    for chunk in chunks:

        f.write(
            json.dumps(
                chunk,
                ensure_ascii=False
            )
            + "\n"
        )

print("Saved:", OUTPUT)

Saved: sentence_overlap_chunks.jsonl


In [ ]:
import json

INPUT = "sentence_overlap_chunks.jsonl"

chunks = []

with open(INPUT, "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print(len(chunks))

327


In [ ]:
PROMPT = """
You are enriching document chunks for retrieval.

Write exactly TWO concise sentences.

Sentence 1:
Describe where this chunk belongs in the document.

Sentence 2:
Summarize the topic of this chunk.

Rules

- Do NOT repeat the chunk.
- Do NOT invent information.
- Maximum 50 words.
- Return ONLY the two sentences.

Chapter:
{chapter}

Section:
{section}

Subsection:
{subsection}

Chunk:

{text}
"""

In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
import torch

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
def contextual_enrichment(chunk):

    prompt = PROMPT.format(
        chapter=chunk["metadata"]["chapter"],
        section=chunk["metadata"]["section"],
        subsection=chunk["metadata"]["subsection"],
        text=chunk["text"]
    )

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.2,
        do_sample=False
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
import json

chunks = []

with open("sentence_overlap_chunks.jsonl","r",encoding="utf-8") as f:

    for line in f:

        chunks.append(json.loads(line))

print(len(chunks))

327


In [ ]:
import torch

def contextual_enrichment_batch(batch_chunks):

    prompts = []

    for chunk in batch_chunks:

        prompt = PROMPT.format(
            chapter=chunk["metadata"]["chapter"],
            section=chunk["metadata"]["section"],
            subsection=chunk["metadata"]["subsection"],
            text=chunk["text"]
        )

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        prompts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        )

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    contexts = []

    input_lengths = inputs["attention_mask"].sum(dim=1)

    for i in range(len(batch_chunks)):

        generated_tokens = outputs[i][input_lengths[i]:]

        context = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        contexts.append(context)

    return contexts

In [ ]:
import torch

def contextual_enrichment_batch(batch_chunks):

    prompts = []

    for chunk in batch_chunks:

        prompt = PROMPT.format(
            chapter=chunk["metadata"]["chapter"],
            section=chunk["metadata"]["section"],
            subsection=chunk["metadata"]["subsection"],
            text=chunk["text"]
        )

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        prompts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        )

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    contexts = []

    input_lengths = inputs["attention_mask"].sum(dim=1)

    for i in range(len(batch_chunks)):

        generated_tokens = outputs[i][input_lengths[i]:]

        context = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        contexts.append(context)

    return contexts

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
print("Device:", next(model.parameters()).device)

CUDA: True
Device: cuda:0


In [ ]:
from tqdm import tqdm

BATCH_SIZE = 8

for i in tqdm(range(0, len(chunks), BATCH_SIZE)):

    batch = chunks[i:i+BATCH_SIZE]

    contexts = contextual_enrichment_batch(batch)

    for chunk, context in zip(batch, contexts):

        chunk["context"] = context

        chunk["enriched_text"] = (
            context
            + "\n\n"
            + chunk["text"]
        )

  0%|          | 0/41 [00:00<?, ?it/s][transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
100%|██████████| 41/41 [24:26<00:00, 35.77s/it]


In [ ]:
for chunk in chunks[:2]:

    print("=" * 80)

    print("Context:")================================================================================
Context:
are correctwen, an AI from Alibaba Cloud. The a knowledgeable and friendly AI
根底CLOCKS
Sentence:

Text:
WHO recommendations for care of the preterm or low birth weight infant ISBN 978-92-4-005826-2 (electronic version) ISBN 978-92-4-005827-9 (print version) © 2022 Some rights reserved.  This work is available under the Creative Commons Attribution-NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/licenses/by-nc-sa/3.0/igo).  Under the terms of this licence, y
================================================================================
Context:$0
    print(chunk["context"])

    print()

    print("Text:")
    print(chunk["text"][:400])

Context:
are correctwen, an AI from Alibaba Cloud. The a knowledgeable and friendly AI
根底CLOCKS
Sentence:

Text:
WHO recommendations for care of the preterm or low birth weight infant ISBN 978-92-4-005826-2 (electronic version) ISBN 978-92-4-005827-9 (print version) © 2022 Some rights reserved.  This work is available under the Creative Commons Attribution-NonCommercial-ShareAlike 3.0 IGO licence (CC BY-NC-SA 3.0 IGO; https://creativecommons.org/licenses/by-nc-sa/3.0/igo).  Under the terms of this licence, y
Context:


Text:
WHO recommendations for care of the preterm or low birth weight infant.  Geneva: ; 2022.  Licence: CC BY-NC-SA 3.0 IGO.  Cataloguing-in-Publication (CIP) data.  CIP data are available at http://apps.who.int/iris.  Sales, rights and licensing.  To purchase WHO publications, see http://apps.who.int/bookorders.  To submit requests for commercial use and queries on rights and licensing, see https://ww


In [ ]:
import json

OUTPUT = "enriched_chunks.jsonl"

with open(OUTPUT, "w", encoding="utf-8") as f:

    for chunk in chunks:

        f.write(
            json.dumps(
                chunk,
                ensure_ascii=False
            )
            + "\n"
        )

print(f"Saved {len(chunks)} chunks -> {OUTPUT}")

Saved 327 chunks -> enriched_chunks.jsonl


In [ ]:
from pprint import pprint

pprint(chunks[70])

{'context': 'Sentence 1: This chunk belongs in the "Evidence and '
            'Recommendations" section, detailing the impact of breastfeeding '
            'on neurodevelopment and hospital stays.\n'
            'Sentence 2: The topic summarizes the minimal impact of '
            'breastfeeding on psychomotor and cognitive neurodevelopment, '
            'along',
 'enriched_text': 'Sentence 1: This chunk belongs in the "Evidence and '
                  'Recommendations" section, detailing the impact of '
                  'breastfeeding on neurodevelopment and hospital stays.\n'
                  'Sentence 2: The topic summarizes the minimal impact of '
                  'breastfeeding on psychomotor and cognitive '
                  'neurodevelopment, along\n'
                  '\n'
                  '**Neurodevelopment** Very-low-certainty evidence from one '
                  'trial with 579 participants indicates minimal impact on '
                  'Griffith quotients for psyc

In [ ]:
import json

with open("enriched_chunks.jsonl", "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f]

required_fields = ["chapter", "section", "subsection", "page"]

missing_count = 0

for i, chunk in enumerate(chunks):
    metadata = chunk.get("metadata", {})

    missing = []

    for field in required_fields:
        value = metadata.get(field)

        if value is None or value == "":
            missing.append(field)

    if missing:
        missing_count += 1
        print("=" * 80)
        print(f"Chunk #{i}")
        print("Missing:", missing)
        print("Metadata:", metadata)
        print("Text preview:", chunk["text"][:200], "...")

print("\n" + "=" * 80)
print(f"Total chunks: {len(chunks)}")
print(f"Chunks with missing metadata: {missing_count}")

Chunk #0
Missing: ['chapter', 'section', 'subsection', 'page']
Metadata: {'chapter': None, 'section': None, 'subsection': None}
Text preview: WHO recommendations for care of the preterm or low birth weight infant ISBN 978-92-4-005826-2 (electronic version) ISBN 978-92-4-005827-9 (print version) © 2022 Some rights reserved.  This work is ava ...
Chunk #1
Missing: ['chapter', 'section', 'subsection', 'page']
Metadata: {'chapter': None, 'section': None, 'subsection': None}
Text preview: WHO recommendations for care of the preterm or low birth weight infant.  Geneva: ; 2022.  Licence: CC BY-NC-SA 3.0 IGO.  Cataloguing-in-Publication (CIP) data.  CIP data are available at http://apps.w ...
Chunk #2
Missing: ['chapter', 'section', 'subsection', 'page']
Metadata: {'chapter': None, 'section': None, 'subsection': None}
Text preview: The mention of sp Your response did not follow my instructions properly.  It's too brief and lacks detail.  Expand significantly on each section, especially where t